In [ ]:
!pip install -q fastai timm transformers scipy tacoreader rasterio


In [ ]:
import sys
from pathlib import Path

REPO_URL = "https://github.com/USERNAME/cloudband.git"  # set your repository URL
REPO_DIR = Path("cloudband")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}

sys.path.insert(0, str(REPO_DIR / "src"))


In [ ]:
from pathlib import Path

import pandas as pd

from cloudband.datasets import cloudsen12 as dataset
from cloudband.datasets.cloudsen12 import iter_samples
from cloudband.eval.metrics import balanced_overall_accuracy
from cloudband.models.ocm import OcmEnsemble, run_ocm_ensemble_phase2
from cloudband.models.swin_upernet import run_swin_upernet_phase2
from cloudband.pipelines.cloudsen12 import per_scene_metric, score_split
from cloudband.train.comparison import compare_per_scene_boa
from cloudband.train.data import build_dataloaders
from cloudband.train.predictor import build_predictor
from cloudband.train.protocol import ocm_shared_protocol, swin_shared_protocol


In [ ]:
def load_split(remote_source, split):
    table = dataset.load_remote(remote_source)
    differences = dataset.verify_split_sizes(table)
    if differences:
        print("split sizes differ from documented ones:", differences)
    return dataset.select(table, split=split)


train_l1c = load_split(dataset.REMOTE_L1C, split="train")
train_l2a = load_split(dataset.REMOTE_L2A, split="train")
train_table = pd.concat([train_l1c, train_l2a], ignore_index=True)

valid_l1c = load_split(dataset.REMOTE_L1C, split="validation")
valid_l2a = load_split(dataset.REMOTE_L2A, split="validation")
valid_table = pd.concat([valid_l1c, valid_l2a], ignore_index=True)

test_table = load_split(dataset.REMOTE_L1C, split="test")

print("train patches:", len(train_table))
print("validation patches:", len(valid_table))
print("test patches:", len(test_table))


In [ ]:
MICRO_BATCH_SIZE = 8  # tune to available GPU memory; effective batch stays
                       # at the protocol's effective_batch_size regardless,
                       # via gradient accumulation

train_valid_dls = build_dataloaders(
    train_table=train_table,
    valid_table=valid_table,
    micro_batch_size=MICRO_BATCH_SIZE,
    num_workers=2,
)


In [ ]:
SEED = 0  # must be one of TrainProtocol's seeds; run once per seed for the
          # full comparison

ocm_protocol = ocm_shared_protocol(learning_rate=1e-4)  # placeholder, the
                                                          # search overwrites it

ocm_runs = run_ocm_ensemble_phase2(train_valid_dls, ocm_protocol, seed=SEED)

for backbone_name, run in ocm_runs.items():
    print(backbone_name)
    print("  winning learning rate:", run.winning_protocol.learning_rate)
    print("  best validation loss:", run.fit_result.best_val_loss)


In [ ]:
swin_protocol = swin_shared_protocol(learning_rate=1e-4)  # placeholder

swin_run = run_swin_upernet_phase2(train_valid_dls, swin_protocol, seed=SEED)

print("winning learning rate:", swin_run.winning_protocol.learning_rate)
print("best validation loss:", swin_run.fit_result.best_val_loss)


In [ ]:
MANIFEST_DIR = Path("manifests")

for run in ocm_runs.values():
    path = run.manifest.write(MANIFEST_DIR / f"{run.manifest.result_name}.json")
    print("wrote", path)

swin_manifest_path = swin_run.manifest.write(
    MANIFEST_DIR / f"{swin_run.manifest.result_name}.json"
)
print("wrote", swin_manifest_path)


In [ ]:
ocm_ensemble_model = OcmEnsemble(
    tuple(run.fit_result.learner.model for run in ocm_runs.values())
)
ocm_predictor = build_predictor(ocm_ensemble_model)
swin_predictor = build_predictor(swin_run.fit_result.learner.model)

ocm_per_scene = score_split(iter_samples(test_table), ocm_predictor)
swin_per_scene = score_split(iter_samples(test_table), swin_predictor)

ocm_boa = per_scene_metric(ocm_per_scene, balanced_overall_accuracy)
swin_boa = per_scene_metric(swin_per_scene, balanced_overall_accuracy)

print(ocm_boa.mean())
print(swin_boa.mean())


In [ ]:
results = compare_per_scene_boa(ocm_boa, swin_boa)

for result in results:
    print(result.experiment)
    print("  pairable scenes:", result.n_pairs)
    print("  p-value:", result.p_value)
    print("  median difference, ocm minus swin:", result.median_difference)
